In [1]:
# ============================================================
# LICENSE KEY TRACKER — Configuration
# ============================================================
# Before running:
#   1. Upload your Excel file to BenefitsPool Lakehouse → Files/
#   2. Set EXCEL_FILE_NAME to match your uploaded file name
#
# Validation uses Microsoft's anonymous product-key redemption endpoint
# (signup.microsoft.com) — no app registration, sign-in, or Key Vault needed.
# ============================================================

EXCEL_FILE_NAME = "License keys 4th batch.xlsx"  # File uploaded to Lakehouse Files
LAKEHOUSE_TABLE = "license_keys"                # Delta table created/refreshed

print("✓ Configuration loaded")
print(f"  Excel file  : {EXCEL_FILE_NAME}")
print(f"  Delta table : {LAKEHOUSE_TABLE}")


StatementMeta(, 3d95e761-a5ee-43fc-88ae-893c5f8a1464, 6, Finished, Available, Finished, False)

✓ Configuration loaded
  Excel file  : License keys 4th batch.xlsx
  Delta table : license_keys


## Step 1 — Ingest Excel → Delta Table
Upload your Excel file to **BenefitsPool Lakehouse → Files/** in the Fabric portal, then run the cell below.  
This creates (or refreshes) the `license_keys` Delta table with all keys and blank validation columns ready for checking.

In [27]:
import pandas as pd
import re
from datetime import datetime
from pyspark.sql.functions import lit
from pyspark.sql.types import StringType

# FORCE_REINGEST = True  → wipe the table and rebuild from EXCEL_FILE_NAME.
# FORCE_REINGEST = False → INCREMENTAL: add only NEW keys from EXCEL_FILE_NAME,
#                          preserving all existing keys and their validation status.
#                          (Upload a new batch, set EXCEL_FILE_NAME, and run this cell.)
FORCE_REINGEST = False

_TRACKING_COLS = ["redemption_status", "check_response", "last_checked",
                  "previous_status", "status_changed_at"]


def _read_excel_keys(file_name: str):
    """Read + clean an Excel file of license keys into a Spark DataFrame with tracking columns."""
    excel_path = f"/lakehouse/default/Files/{file_name}"
    print(f"Reading: {excel_path}")
    df = pd.read_excel(excel_path, engine="openpyxl")
    print(f"Raw shape: {df.shape}  |  Columns: {list(df.columns)}")

    def clean_col(name: str) -> str:
        name = str(name).strip()
        name = re.sub(r"[&]", "and", name)
        name = re.sub(r"[^a-zA-Z0-9\s]", "", name)
        name = re.sub(r"\s+", "_", name.strip())
        return name.lower()

    df.columns = [clean_col(c) for c in df.columns]
    print("Cleaned columns:", list(df.columns))
    if "license_key" not in df.columns:
        raise ValueError(f"'License Key' column not found after cleaning. Got: {list(df.columns)}")

    before = len(df)
    df = df[df["license_key"].notna() & (df["license_key"].astype(str).str.strip() != "")]
    print(f"Rows after removing blank keys: {len(df)}  (removed {before - len(df)})")

    df_str = df.astype(str).where(df.notna(), None)
    sdf = spark.createDataFrame(df_str)
    for c in _TRACKING_COLS:
        sdf = sdf.withColumn(c, lit(None).cast(StringType()))
    sdf = sdf.withColumn("inserted_at", lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S")))
    # De-duplicate within the incoming file itself
    return sdf.dropDuplicates(["license_key"])


new_sdf = _read_excel_keys(EXCEL_FILE_NAME)

if FORCE_REINGEST or (not spark.catalog.tableExists(LAKEHOUSE_TABLE)):
    # ── Fresh build (or forced full rebuild) ─────────────────────────────
    (new_sdf.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(LAKEHOUSE_TABLE))
    spark.catalog.refreshTable(LAKEHOUSE_TABLE)
    total = spark.table(LAKEHOUSE_TABLE).select("license_key").distinct().count()
    print(f"\n✓ Delta table '{LAKEHOUSE_TABLE}' built fresh — {total} distinct keys.")
else:
    # ── Incremental: insert only keys NOT already present ────────────────
    # Existing keys keep their validation status; only brand-new keys are added.
    table_cols = [f.name for f in spark.table(LAKEHOUSE_TABLE).schema.fields]
    for c in table_cols:
        if c not in new_sdf.columns:
            new_sdf = new_sdf.withColumn(c, lit(None).cast(StringType()))
    new_sdf = new_sdf.select(table_cols)

    before_cnt = spark.table(LAKEHOUSE_TABLE).select("license_key").distinct().count()
    new_sdf.createOrReplaceTempView("_incoming_keys")
    spark.sql(f"""
        MERGE INTO {LAKEHOUSE_TABLE} AS t
        USING _incoming_keys AS s
            ON t.license_key = s.license_key
        WHEN NOT MATCHED THEN INSERT *
    """)
    spark.catalog.refreshTable(LAKEHOUSE_TABLE)
    after_cnt = spark.table(LAKEHOUSE_TABLE).select("license_key").distinct().count()
    print(f"\n✓ Incremental ingest of '{EXCEL_FILE_NAME}':")
    print(f"   New keys added : {after_cnt - before_cnt}")
    print(f"   Total keys now : {after_cnt}")

print("\nStatus breakdown:")
display(spark.sql(f"""
    SELECT COALESCE(redemption_status, 'not_checked') AS redemption_status,
           COUNT(*) AS n
    FROM {LAKEHOUSE_TABLE}
    GROUP BY redemption_status ORDER BY n DESC
"""))


StatementMeta(, e28cf6d6-3bf4-4a1c-b1c3-521bea0ed71d, 9, Finished, Available, Finished, False)

Reading: /lakehouse/default/Files/License keys 4th batch.xlsx
Raw shape: (17, 8)  |  Columns: ['License Description', 'Program Name & Specialization', 'Quantity', 'Status', 'License Key', 'Activate by', 'Date Issued', 'MPN Region']
Cleaned columns: ['license_description', 'program_name_and_specialization', 'quantity', 'status', 'license_key', 'activate_by', 'date_issued', 'mpn_region']
Rows after removing blank keys: 17  (removed 0)

✓ Incremental ingest of 'License keys 4th batch.xlsx':
   New keys added : 17
   Total keys now : 345

Status breakdown:


SynapseWidget(Synapse.DataFrame, 16667c6a-c6c7-4158-b7cc-7bc043a68659)

## Step 2 — Load Validation Functions
Run this cell once per session to define the key-checking helper.  
`check_key_status()` calls the anonymous `validatePrepaidKeys` endpoint on `signup.microsoft.com` and returns a structured status for each key — no sign-in required.

In [15]:
import requests
import re
import time
import json
import uuid
from typing import Tuple, Optional
from urllib.parse import urlparse

# ══════════════════════════════════════════════════════════════════════════════
# Status icons (used by the batch + summary cells)
# ══════════════════════════════════════════════════════════════════════════════
_STATUS_ICONS = {
    "available":   "✅",
    "redeemed":    "🔴",
    "expired":     "⚠️",
    "invalid":     "❌",
    "error":       "🟠",
    "ratelimited": "⏳",
    "unknown":     "❓",
}

# Kept for backward compatibility with older cells; not required anymore.
_session: Optional[requests.Session] = None


# ══════════════════════════════════════════════════════════════════════════════
# Anonymous product-key lookup  (signup.microsoft.com redemption "About you" step)
# ══════════════════════════════════════════════════════════════════════════════
_VALIDATE_URL   = "https://signup.microsoft.com/api/signupservice/validatePrepaidKeys"
_SIGNUP_REFERER = "https://signup.microsoft.com/get-started/setupKey"
_prepaid_session: Optional[requests.Session] = None


def _get_prepaid_session() -> requests.Session:
    """
    A browser-like session warmed against signup.microsoft.com so the
    anonymous validatePrepaidKeys endpoint accepts our requests.
    The redemption 'About you' step validates keys BEFORE sign-in, so no
    ESTSAUTH cookie or bearer token is required.
    """
    global _prepaid_session
    if _prepaid_session is not None:
        return _prepaid_session
    s = requests.Session()
    s.headers.update({
        "User-Agent":      "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                           "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
        "Accept":          "application/json, text/plain, */*",
        "Accept-Language": "en-US,en;q=0.9",
        "Origin":          "https://signup.microsoft.com",
        "Referer":         _SIGNUP_REFERER,
    })
    try:
        s.get("https://signup.microsoft.com/productkeystart",
              timeout=30, allow_redirects=True)
    except Exception:
        pass   # cookies are best-effort; the endpoint is anonymous
    _prepaid_session = s
    return s


def _strip_html(text: str) -> str:
    """Turn the product-description HTML into a plain one-line label."""
    clean = re.sub(r"<[^>]+>", " ", text or "")
    return re.sub(r"\s+", " ", clean.replace("\xa0", " ")).strip()


def check_key_status(product_key: str, session: requests.Session = None,
                     description: str = "") -> Tuple[str, str]:
    """
    Look up ONE product key against Microsoft's anonymous redemption
    validation endpoint — the same call the 'Redeem your product key'
    page fires on its first ('About you') step.

        POST https://signup.microsoft.com/api/signupservice/validatePrepaidKeys
             ?culture=en-us&api-version=1&client-request-id=<guid>
        body: {"keys": ["XXXXX-XXXXX-XXXXX-XXXXX-XXXXX"]}

    Interpretation (tokenStatus[0]):
        isTokenValid = true   → available  (not yet redeemed; product shown)
        statusCode   = 5      → redeemed   ("already been used")
        expired text          → expired
        anything else invalid → redeemed   (business rule: not-found = used)

    `description` and `session` are accepted for backward compatibility but
    are not required — the endpoint is self-describing and anonymous.
    Returns (status, message).
    """
    product_key = str(product_key).strip()
    if not product_key or product_key.upper() in ("NAN", "NONE", ""):
        return ("invalid", "Empty or null key — skipped")

    sess   = session if isinstance(session, requests.Session) else _get_prepaid_session()
    params = {"culture": "en-us", "api-version": "1",
              "client-request-id": str(uuid.uuid4())}

    try:
        resp = sess.post(
            _VALIDATE_URL, params=params, json={"keys": [product_key]},
            headers={"Content-Type": "application/json",
                     "Accept": "application/json, text/plain, */*",
                     "Origin": "https://signup.microsoft.com",
                     "Referer": _SIGNUP_REFERER},
            timeout=30,
        )
    except requests.Timeout:
        return ("error", "Request timed out (30 s)")
    except requests.ConnectionError as exc:
        return ("error", f"Connection error: {str(exc)[:120]}")
    except Exception as exc:
        return ("error", f"Request failed: {str(exc)[:160]}")
    if resp.status_code == 429:
        ra = resp.headers.get("Retry-After")
        return ("ratelimited", f"HTTP 429 TooManyRequests (Retry-After={ra})")
    if resp.status_code != 200:
        return ("error", f"HTTP {resp.status_code} from validatePrepaidKeys")
    try:
        data = resp.json()
    except ValueError:
        return ("error", "Non-JSON response — endpoint may need a warmed browser session")

    val      = (data or {}).get("prepaidKeysValidationResult") or {}
    statuses = val.get("tokenStatus") or []
    if not statuses:
        return ("unknown",
                f"No tokenStatus returned (responseCode={data.get('responseCode')})")

    ts       = statuses[0]
    is_valid = bool(ts.get("isTokenValid"))
    msg      = (ts.get("tokenValidationMessage") or "").strip()
    code     = ts.get("statusCode")
    details  = ts.get("tokenDetails") or {}
    seats    = details.get("seatCount", 0)
    offer    = details.get("offerID", "")

    # ── Valid → available for redemption ─────────────────────────────────
    if is_valid:
        desc_res = (data or {}).get("prepaidKeysDescriptionResult") or {}
        product  = _strip_html(desc_res.get("description") or desc_res.get("summary") or "")
        label    = product or msg or "valid product key"
        extra    = f" ({seats} seats)" if seats else ""
        tail     = (f" · offer {offer}"
                    if offer and offer != "00000000-0000-0000-0000-000000000000" else "")
        return ("available", f"Available — {label}{extra}{tail}")

    # ── Not valid → classify ─────────────────────────────────────────────
    low = msg.lower()
    if "expired" in low or "no longer valid" in low or "past" in low:
        return ("expired", msg or "Key has expired / past activation deadline")
    if code == 5 or "already been used" in low or "already redeemed" in low:
        return ("redeemed", msg or "Key has already been redeemed")
    # Business rule: any other 'not found / not valid' outcome = already used
    return ("redeemed", msg or f"Key not usable (statusCode {code}) — treated as redeemed")


print("✓ Validation functions loaded  (anonymous validatePrepaidKeys lookup)")
print("  check_key_status(key)              →  (status, message) for one key")

print("  _get_prepaid_session()             →  warmed signup.microsoft.com session")

print("  No sign-in required — the redemption 'About you' step is anonymous.")

StatementMeta(, d5e06672-4355-4e0d-914d-6d18aa77ad6d, 11, Finished, Available, Finished, False)

✓ Validation functions loaded  (anonymous validatePrepaidKeys lookup)
  check_key_status(key)              →  (status, message) for one key
  _get_prepaid_session()             →  warmed signup.microsoft.com session
  No sign-in required — the redemption 'About you' step is anonymous.


## Step 3 — Run / Re-verify Validation (rate-limit friendly)
Checks up to `MAX_KEYS_PER_RUN` keys per run and checkpoints each result to the Delta table immediately. Microsoft rate-limits the endpoint, so keep the per-run count low and **schedule this notebook** (e.g. every 15 minutes) to work through keys over time.

**Detecting unused keys:** a key is either `available` (NOT yet redeemed/used) or `redeemed` (already used). To catch keys that were marked redeemed but were never actually consumed, set `RECHECK_STATUSES = ["redeemed"]` — each scheduled run re-verifies redeemed keys (oldest-checked first) and **flags any that come back `available`** as reclaimable. Set `RECHECK_STATUSES = []` to only fill in never-checked keys.

Re-running is always safe and fully resumable.

In [ ]:
import pandas as pd
import time
from datetime import datetime

# ── Tuning ───────────────────────────────────────────────────────────────────
# signup.microsoft.com rate-limits (429) if queried too often, so we check only a
# few keys per run and let a SCHEDULE space the runs out. Every result is
# checkpointed to the Delta table immediately, so the job is fully resumable.
#
# A key is either "available" (NOT yet redeemed/used) or "redeemed" (already used).
# RECHECK_STATUSES lists which existing statuses to re-verify each run so we can
# catch flips in BOTH directions (redeemed→available = reclaimable unused key;
# available→redeemed = key was just consumed).
RECHECK_ALL      = False                       # True = re-validate every key from scratch
RECHECK_STATUSES = ["available"]               # only re-verify still-available keys; NEVER re-check
                                               # utilized (redeemed) keys. Use [] to stop all
                                               # re-checks, or ["redeemed", "available"] to watch both.
MAX_KEYS_PER_RUN = 10                          # keys to check per run (under the ~14 burst limit)
BASE_DELAY       = 20                          # seconds between checks WITHIN a run (only if >1)
STOP_AFTER       = 2                           # stop early after this many consecutive 429s

# Build the "which keys to check" filter
if RECHECK_ALL:
    _status_filter = "TRUE"
else:
    _clauses = [
        "redemption_status IS NULL",
        "redemption_status IN ('unknown', 'error', 'ratelimited')",
    ]
    if RECHECK_STATUSES:
        _quoted = ", ".join(f"'{s}'" for s in RECHECK_STATUSES)
        _clauses.append(f"redemption_status IN ({_quoted})")
    _status_filter = "(" + " OR ".join(_clauses) + ")"

# Ensure change-tracking columns exist (idempotent — safe on every run)
_existing_cols = [f.name for f in spark.table(LAKEHOUSE_TABLE).schema.fields]
for _c in ("previous_status", "status_changed_at"):
    if _c not in _existing_cols:
        spark.sql(f"ALTER TABLE {LAKEHOUSE_TABLE} ADD COLUMNS ({_c} STRING)")

# Total in scope for checking (for reporting)
total_pending = spark.sql(f"""
    SELECT COUNT(DISTINCT license_key) AS n FROM {LAKEHOUSE_TABLE}
    WHERE license_key IS NOT NULL
      AND TRIM(license_key) NOT IN ('', 'nan', 'None')
      AND {_status_filter}
""").collect()[0]["n"]

# This run's slice: oldest-checked / never-checked first, capped per run
pending_df = spark.sql(f"""
    SELECT license_key,
           MAX(license_description) AS license_description,
           MAX(redemption_status)   AS prev_status
    FROM   {LAKEHOUSE_TABLE}
    WHERE  license_key IS NOT NULL
      AND  TRIM(license_key) NOT IN ('', 'nan', 'None')
      AND  {_status_filter}
    GROUP BY license_key
    ORDER BY MIN(last_checked) ASC NULLS FIRST,
             MIN(inserted_at)  ASC
    LIMIT {MAX_KEYS_PER_RUN}
""").toPandas()

this_run = len(pending_df)
print(f"Keys in scope : {total_pending}   |   checking this run : {this_run}")
print("─" * 60)


def _merge_one(rec: dict) -> None:
    """
    Checkpoint a single key result into the Delta table immediately.
    Records status flips: when the new status differs from the stored one,
    previous_status + status_changed_at are set so changes are queryable.
    """
    sdf = spark.createDataFrame(pd.DataFrame([rec]))
    sdf.createOrReplaceTempView("_one_update")
    spark.sql(f"""
        MERGE INTO {LAKEHOUSE_TABLE} AS t
        USING _one_update AS s
            ON t.license_key = s.license_key
        WHEN MATCHED THEN UPDATE SET
            t.previous_status   = CASE
                WHEN t.redemption_status IS NOT NULL
                     AND t.redemption_status <> s.redemption_status
                THEN t.redemption_status ELSE t.previous_status END,
            t.status_changed_at = CASE
                WHEN t.redemption_status IS NOT NULL
                     AND t.redemption_status <> s.redemption_status
                THEN s.last_checked ELSE t.status_changed_at END,
            t.redemption_status = s.redemption_status,
            t.check_response    = s.check_response,
            t.last_checked      = s.last_checked
    """)


if this_run == 0:
    print("🎉 Nothing in scope — all keys already checked for this configuration.")
else:
    counts, consec_limit, done, flips = {}, 0, 0, []

    for i, row in pending_df.iterrows():
        key  = str(row["license_key"]).strip()
        desc = str(row.get("license_description", "") or "")
        prev = str(row.get("prev_status") or "")

        status, message = check_key_status(key, description=desc)

        # Rate limited → defer (do NOT retry in-session; that extends the block).
        if status == "ratelimited":
            consec_limit += 1
            print(f"  ⏳  {key[-15:]:>15}  →  rate-limited (deferred — try again later)")
            if consec_limit >= STOP_AFTER:
                print("\n⏸  Rate-limited. Stopping — re-run after the schedule interval.")
                break
            continue
        consec_limit = 0

        checked_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        _merge_one({
            "license_key":       key,
            "redemption_status": status,
            "check_response":    message[:500],
            "last_checked":      checked_at,
        })
        done += 1
        counts[status] = counts.get(status, 0) + 1
        icon = _STATUS_ICONS.get(status, "❓")

        # Flag status changes in either direction
        changed = bool(prev) and status != prev
        flag = ""
        if changed:
            flag = f"   ⚠ CHANGED: {prev} → {status}"
            flips.append((key, desc, prev, status))
        print(f"  {icon}  {key[-15:]:>15}  →  {status:<11}  ({desc[:34]}){flag}")

        if this_run > 1:
            time.sleep(BASE_DELAY)

    # ── Pass summary ─────────────────────────────────────────────────────
    remaining = max(total_pending - done, 0)
    print("─" * 60)
    print(f"✓ Checkpointed {done} key(s) this run  ·  {remaining} still in scope")
    for st, cnt in sorted(counts.items(), key=lambda x: -x[1]):
        print(f"   {_STATUS_ICONS.get(st, '❓')}  {st:<12} : {cnt}")

    if flips:
        print("\n🔄  Status changes this run:")
        for k, d, p, s in flips:
            note = "  ← reclaimable (unused)" if (p == "redeemed" and s == "available") else ""
            print(f"     {p} → {s}   {k}   ({d[:34]}){note}")

    if remaining:
        print("\n   Schedule this notebook to re-run periodically to work through the rest.")
    else:
        print("\n   🎉 All keys in scope have been checked.")


StatementMeta(, d5e06672-4355-4e0d-914d-6d18aa77ad6d, 12, Finished, Available, Finished, False)

Keys in scope : 116   |   checking this run : 10
────────────────────────────────────────────────────────────
  🔴  H6B-Q2VT2-XTQQH  →  redeemed     (M365 E5 )
  🔴  GXF-CVYPG-Y96JF  →  redeemed     (Microsoft Dynamics 365 - Customer )
  🔴  MC4-63R7F-37CMK  →  redeemed     (Microsoft 365 E5)
  🔴  B3K-9DWVB-MY7DH  →  redeemed     (M365 BP )
  🔴  XK3-TMGXD-YBKVX  →  redeemed     (Dynamics 365 Partner Sandbox – Sal)
  ✅  VXF-YJ8V4-CR3VM  →  available    (Microsoft Project Online Project P)
  ✅  KG4-622MX-8QHD2  →  available    (Microsoft Project Online Essential)
  ✅  TT6-D2MY6-YHCMK  →  available    (Dynamics 365 Business Central Prem)
  ✅  B4D-RW7FQ-9D9VX  →  available    (Microsoft 365 E5)
  🔴  PPM-MY7Y3-CBBQ9  →  redeemed     (Dynamics 365 Partner Sandbox – Sal)
────────────────────────────────────────────────────────────
✓ Checkpointed 10 key(s) this run  ·  106 still in scope
   🔴  redeemed     : 6
   ✅  available    : 4

   Schedule this notebook to re-run periodically to work throug

## Step 4 — View Results
Run at any time to see the current state of all keys in the warehouse.  
The summary table and the detailed grid are both sourced directly from the live Delta table.

In [ ]:
# ── Status breakdown ─────────────────────────────────────────────────────────
print("=" * 55)
print("  LICENSE KEY STATUS SUMMARY")
print("=" * 55)

summary = spark.sql(f"""
    SELECT
        COALESCE(redemption_status, 'not_checked') AS redemption_status,
        COUNT(*)                                   AS key_count,
        ROUND(COUNT(*) * 100.0
              / SUM(COUNT(*)) OVER (), 1)          AS pct_of_total
    FROM   {LAKEHOUSE_TABLE}
    GROUP  BY redemption_status
    ORDER  BY key_count DESC
""")
display(summary)

# ── Full detail grid ─────────────────────────────────────────────────────────
print("\nAll keys — sorted by status priority then description:")
display(spark.sql(f"""
    SELECT
        license_description,
        program_name_and_specialization,
        license_key,
        quantity,
        activate_by,
        mpn_region,
        redemption_status,
        previous_status,
        status_changed_at,
        check_response,
        last_checked
    FROM   {LAKEHOUSE_TABLE}
    ORDER BY
        CASE redemption_status
            WHEN 'available'   THEN 1
            WHEN 'redeemed'    THEN 2
            WHEN 'expired'     THEN 3
            WHEN 'invalid'     THEN 4
            WHEN 'error'       THEN 5
            ELSE                    6
        END,
        license_description
"""))

# ── Available keys only (quick export view) ───────────────────────────────────
available = spark.sql(f"""
    SELECT license_description, program_name_and_specialization,
           license_key, activate_by, mpn_region
    FROM   {LAKEHOUSE_TABLE}
    WHERE  redemption_status = 'available'
    ORDER  BY activate_by
""")
avail_count = available.count()
print(f"\n✅  Available (unredeemed) keys: {avail_count}")
if avail_count > 0:
    display(available)

# ── Keys whose status has flipped (redeemed ↔ available, etc.) ────────────────
changed = spark.sql(f"""
    SELECT license_description, license_key,
           previous_status,
           redemption_status AS current_status,
           status_changed_at, last_checked
    FROM   {LAKEHOUSE_TABLE}
    WHERE  previous_status IS NOT NULL
      AND  previous_status <> redemption_status
    ORDER  BY status_changed_at DESC
""")
chg_count = changed.count()
print(f"\n🔄  Keys that changed status since first check: {chg_count}")
if chg_count > 0:
    display(changed)


StatementMeta(, 9aeb42c1-4a53-4b87-99a0-f73235509a7c, 9, Finished, Available, Finished, False)

  LICENSE KEY STATUS SUMMARY


SynapseWidget(Synapse.DataFrame, 3c45966a-57c0-4a8d-b0d0-09df700327e8)


All keys — sorted by status priority then description:


SynapseWidget(Synapse.DataFrame, 3b0a90c9-2e8a-43a4-8b38-e17bd0f2bd37)


✅  Available (unredeemed) keys: 75


SynapseWidget(Synapse.DataFrame, 7b546bc8-6efd-4ede-872d-e483c67a6b26)


🔄  Keys that changed status since first check: 1


SynapseWidget(Synapse.DataFrame, bc9f92f7-a1d9-423e-876e-1bf62e9aa75f)

In [ ]:
# ── Available vs Redeemed licenses ────────────────────────────────────────────
# Two clean tables: keys ready to use (available) and keys already used (redeemed).
print("=" * 60)
print("  AVAILABLE  vs  REDEEMED  LICENSES")
print("=" * 60)

status_view = spark.sql(f"""
    SELECT
        redemption_status,
        license_description,
        program_name_and_specialization,
        license_key,
        quantity,
        activate_by,
        mpn_region,
        last_checked
    FROM   {LAKEHOUSE_TABLE}
    WHERE  redemption_status IN ('available', 'redeemed')
    ORDER BY
        CASE redemption_status WHEN 'available' THEN 1 ELSE 2 END,
        license_description
""")

avail_tbl  = status_view.filter("redemption_status = 'available'").drop("redemption_status")
redeem_tbl = status_view.filter("redemption_status = 'redeemed'").drop("redemption_status")

print(f"✅ Available (unredeemed): {avail_tbl.count()}")
print(f"🔴 Redeemed (used)      : {redeem_tbl.count()}")

print("\n✅ AVAILABLE licenses (ready to assign):")
display(avail_tbl)

print("\n🔴 REDEEMED licenses (already used):")
display(redeem_tbl)


StatementMeta(, 14134cb0-598f-43d1-ab0d-96645c0e7eb3, 4, Finished, Available, Finished, False)

  AVAILABLE  vs  REDEEMED  LICENSES
✅ Available (unredeemed): 75
🔴 Redeemed (used)      : 278

✅ AVAILABLE licenses (ready to assign):


SynapseWidget(Synapse.DataFrame, 2ee2adae-46e8-41ee-8976-f7b8f4f73cc4)


🔴 REDEEMED licenses (already used):


SynapseWidget(Synapse.DataFrame, 15d73f61-1201-442a-9b5a-8a9a562e2f1a)

In [2]:
# ── Export available keys to a single CSV in the Lakehouse (for download) ──────
export_pdf = spark.sql(f"""
    SELECT license_description, program_name_and_specialization,
           license_key, quantity, activate_by, date_issued, mpn_region, last_checked
    FROM   {LAKEHOUSE_TABLE}
    WHERE  redemption_status = 'available'
    ORDER  BY license_description
""").toPandas()

out_path = "/lakehouse/default/Files/availablekeys.csv"
export_pdf.to_csv(out_path, index=False)

print(f"✓ Exported {len(export_pdf)} available keys →  Files/availablekeys.csv")
print("\nTo download to your PC:")
print("  • Fabric portal:  BenefitsPool Lakehouse → Files → availablekeys.csv → (…) → Download")
print("  • VS Code:        Fabric/OneLake explorer → BenefitsPool → Files → right-click availablekeys.csv → Download")


StatementMeta(, 3d95e761-a5ee-43fc-88ae-893c5f8a1464, 7, Finished, Available, Finished, False)

✓ Exported 75 available keys →  Files/availablekeys.csv

To download to your PC:
  • Fabric portal:  BenefitsPool Lakehouse → Files → availablekeys.csv → (…) → Download
  • VS Code:        Fabric/OneLake explorer → BenefitsPool → Files → right-click availablekeys.csv → Download
